In [1]:
import pandas as pd

# Load only the small tables fully; peek at the big ones
purchase_prices = pd.read_csv("purchase_prices.csv")
vendor_invoice  = pd.read_csv("vendor_invoice.csv")

# Big files: read just the first 10,000 rows so it's fast
purchases = pd.read_csv("purchases.csv", nrows=10000)
sales     = pd.read_csv("sales.csv", nrows=10000)

# For each, look at: columns, a few rows, and size
for name, df in [("purchase_prices", purchase_prices),
                 ("vendor_invoice", vendor_invoice),
                 ("purchases", purchases),
                 ("sales", sales)]:
    print(f"\n===== {name} =====")
    print("Columns:", list(df.columns))
    display(df.head(3))


===== purchase_prices =====
Columns: ['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification', 'PurchasePrice', 'VendorNumber', 'VendorName']


,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP



===== vendor_invoice =====
Columns: ['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate', 'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval']


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,NaN
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,NaN
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,NaN



===== purchases =====
Columns: ['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber', 'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate', 'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification']


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1



===== sales =====
Columns: ['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity', 'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification', 'ExciseTax', 'VendorNo', 'VendorName']


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY


In [2]:
%run ingestion_db.py

NameError: name 'null' is not defined

In [ ]:
%run ingestion_db.py

In [ ]:
import pandas as pd, os, time
from sqlalchemy import create_engine

engine = create_engine("sqlite:///inventory.db")
big_files = {"sales.csv", "purchases.csv"}     # these get chunked
start = time.time()

for file in os.listdir('.'):
    if not file.endswith('.csv'):
        continue
    table = file[:-4]
    if file in big_files:
        first = True
        rows = 0
        for chunk in pd.read_csv(file, chunksize=100_000):
            chunk.to_sql(table, con=engine,
                         if_exists="replace" if first else "append", index=False)
            first = False
            rows += len(chunk)
            print(f"  {table}: {rows:,} rows", end="\r")
        print(f"  {table}: {rows:,} rows  ✅")
    else:
        pd.read_csv(file).to_sql(table, con=engine, if_exists="replace", index=False)
        print(f"  {table}: loaded ✅")

print(f"\nAll done in {(time.time()-start)/60:.2f} min")

In [6]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine("sqlite:///inventory.db")

query = """
SELECT
    VendorNumber,
    VendorName,
    Brand,
    SUM(Quantity) AS TotalPurchaseQuantity,
    SUM(Dollars)  AS TotalPurchaseDollars
FROM purchases
WHERE PurchasePrice > 0
GROUP BY VendorNumber, VendorName, Brand
"""
purchase_summary = pd.read_sql(query, engine)
print("Rows before (raw purchases): 2,372,474")
print("Rows after grouping:", len(purchase_summary))
purchase_summary.head()

Rows before (raw purchases): 2,372,474
Rows after grouping: 10692


,VendorNumber,VendorName,Brand,TotalPurchaseQuantity,TotalPurchaseDollars
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",90085,8,190.88
1,2,"IRA GOLDMAN AND WILLIAMS, LLP",90609,320,5440.00
2,54,AAPER ALCOHOL & CHEMICAL CO,990,1,105.07
3,60,ADAMBA IMPORTS INTL INC,771,39,446.16
4,60,ADAMBA IMPORTS INTL INC,3401,6,66.60


In [7]:
query = """
SELECT
    VendorNo,
    Brand,
    SUM(SalesQuantity) AS TotalSalesQuantity,
    SUM(SalesDollars)  AS TotalSalesDollars,
    SUM(ExciseTax)     AS TotalExciseTax
FROM sales
GROUP BY VendorNo, Brand
"""
sales_summary = pd.read_sql(query, engine)
print("Rows before (raw sales): 12,825,363")
print("Rows after grouping:", len(sales_summary))
sales_summary.head()

Rows before (raw sales): 12,825,363
Rows after grouping: 11272


,VendorNo,Brand,TotalSalesQuantity,TotalSalesDollars,TotalExciseTax
0,2,90085,18,665.82,2.00
1,2,90609,24,599.76,0.52
2,60,771,47,704.53,37.01
3,60,3979,3931,66871.69,7224.06
4,105,2529,12,359.88,9.44


In [8]:
query = """
SELECT
    VendorNumber,
    SUM(Freight) AS FreightCost
FROM vendor_invoice
GROUP BY VendorNumber
"""
freight_summary = pd.read_sql(query, engine)
print("Rows:", len(freight_summary))
freight_summary.head()

Rows: 126


,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19


In [9]:
query = """
WITH FreightSummary AS (
    SELECT VendorNumber, SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT
        p.VendorNumber, p.VendorName, p.Brand, p.Description,
        p.PurchasePrice,
        pp.Price  AS ActualPrice,
        pp.Volume,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars)  AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description,
             p.PurchasePrice, pp.Price, pp.Volume
),
SalesSummary AS (
    SELECT
        VendorNo, Brand,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(SalesDollars)  AS TotalSalesDollars,
        SUM(SalesPrice)    AS TotalSalesPrice,
        SUM(ExciseTax)     AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)
SELECT
    ps.VendorNumber, ps.VendorName, ps.Brand, ps.Description,
    ps.PurchasePrice, ps.ActualPrice, ps.Volume,
    ps.TotalPurchaseQuantity, ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity, ss.TotalSalesDollars, ss.TotalSalesPrice, ss.TotalExciseTax,
    fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary  ss ON ps.VendorNumber = ss.VendorNo AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs ON ps.VendorNumber = fs.VendorNumber
"""
vendor_summary = pd.read_sql(query, engine)
print("Final shape:", vendor_summary.shape)
vendor_summary.head()

Final shape: (10692, 14)


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",90085,Ch Lilian 09 Ladouys St Este,23.86,36.99,750,8,190.88,18.0,665.82,295.92,2.00,27.08
1,2,"IRA GOLDMAN AND WILLIAMS, LLP",90609,Flavor Essence Variety 5 Pak,17.00,24.99,162.5,320,5440.00,24.0,599.76,449.82,0.52,27.08
2,54,AAPER ALCOHOL & CHEMICAL CO,990,Ethyl Alcohol 200 Proof,105.07,134.49,3750,1,105.07,NaN,NaN,NaN,NaN,0.48
3,60,ADAMBA IMPORTS INTL INC,771,Bak's Krupnik Honey Liqueur,11.44,14.99,750,39,446.16,47.0,704.53,494.67,37.01,367.52
4,60,ADAMBA IMPORTS INTL INC,3401,Vesica Vodka,11.10,14.99,1750,6,66.60,NaN,NaN,NaN,NaN,367.52


In [10]:
# How many NaNs do we have, and where?
print("Missing values per column:")
print(vendor_summary.isnull().sum())

# A brand never sold = 0 sales, not "unknown" -> replace NaN with 0
vendor_summary.fillna(0, inplace=True)

print("\nAfter fillna — missing values:")
print(vendor_summary.isnull().sum())

Missing values per column:
VendorNumber               0
VendorName                 0
Brand                      0
Description                0
PurchasePrice              0
ActualPrice                0
Volume                     0
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesQuantity       178
TotalSalesDollars        178
TotalSalesPrice          178
TotalExciseTax           178
FreightCost                0
dtype: int64

After fillna — missing values:
VendorNumber             0
VendorName               0
Brand                    0
Description              0
PurchasePrice            0
ActualPrice              0
Volume                   0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalSalesPrice          0
TotalExciseTax           0
FreightCost              0
dtype: int64


In [11]:
# MASTER table = everything (save this to the database, lose nothing)
vendor_summary          # all 10,692 rows, including the 178 never-sold

# WORKING copy = only brands that actually sold (use for profit/margin analysis)
sales_df = vendor_summary[vendor_summary["TotalSalesDollars"] > 0].copy()
print("Master rows:", len(vendor_summary))
print("Rows with real sales:", len(sales_df))

Master rows: 10692
Rows with real sales: 10514


In [12]:
import numpy as np

vendor_summary["GrossProfit"]          = vendor_summary["TotalSalesDollars"] - vendor_summary["TotalPurchaseDollars"]
vendor_summary["ProfitMargin"]         = vendor_summary["GrossProfit"] / vendor_summary["TotalSalesDollars"].replace(0, np.nan) * 100
vendor_summary["StockTurnover"]        = vendor_summary["TotalSalesQuantity"] / vendor_summary["TotalPurchaseQuantity"].replace(0, np.nan)
vendor_summary["SalesToPurchaseRatio"] = vendor_summary["TotalSalesDollars"] / vendor_summary["TotalPurchaseDollars"].replace(0, np.nan)
vendor_summary.fillna(0, inplace=True)

# recreate the working copy AFTER metrics exist, so it includes the new columns
sales_df = vendor_summary[vendor_summary["TotalSalesDollars"] > 0].copy()

print("Columns now:", list(vendor_summary.columns))
vendor_summary.head()

Columns now: ['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice', 'ActualPrice', 'Volume', 'TotalPurchaseQuantity', 'TotalPurchaseDollars', 'TotalSalesQuantity', 'TotalSalesDollars', 'TotalSalesPrice', 'TotalExciseTax', 'FreightCost', 'GrossProfit', 'ProfitMargin', 'StockTurnover', 'SalesToPurchaseRatio']


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",90085,Ch Lilian 09 Ladouys St Este,23.86,36.99,750,8,190.88,18.0,665.82,295.92,2.00,27.08,474.94,71.331591,2.250000,3.488160
1,2,"IRA GOLDMAN AND WILLIAMS, LLP",90609,Flavor Essence Variety 5 Pak,17.00,24.99,162.5,320,5440.00,24.0,599.76,449.82,0.52,27.08,-4840.24,-807.029478,0.075000,0.110250
2,54,AAPER ALCOHOL & CHEMICAL CO,990,Ethyl Alcohol 200 Proof,105.07,134.49,3750,1,105.07,0.0,0.00,0.00,0.00,0.48,-105.07,0.000000,0.000000,0.000000
3,60,ADAMBA IMPORTS INTL INC,771,Bak's Krupnik Honey Liqueur,11.44,14.99,750,39,446.16,47.0,704.53,494.67,37.01,367.52,258.37,36.672675,1.205128,1.579097
4,60,ADAMBA IMPORTS INTL INC,3401,Vesica Vodka,11.10,14.99,1750,6,66.60,0.0,0.00,0.00,0.00,367.52,-66.60,0.000000,0.000000,0.000000


In [13]:
vendor_summary.to_sql("vendor_sales_summary", con=engine, if_exists="replace", index=False)
print("✅ Saved vendor_sales_summary:", vendor_summary.shape)

# prove it's really in the database now
check = pd.read_sql("SELECT * FROM vendor_sales_summary LIMIT 3", engine)
check

✅ Saved vendor_sales_summary: (10692, 18)


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",90085,Ch Lilian 09 Ladouys St Este,23.86,36.99,750,8,190.88,18.0,665.82,295.92,2.00,27.08,474.94,71.331591,2.250,3.48816
1,2,"IRA GOLDMAN AND WILLIAMS, LLP",90609,Flavor Essence Variety 5 Pak,17.00,24.99,162.5,320,5440.00,24.0,599.76,449.82,0.52,27.08,-4840.24,-807.029478,0.075,0.11025
2,54,AAPER ALCOHOL & CHEMICAL CO,990,Ethyl Alcohol 200 Proof,105.07,134.49,3750,1,105.07,0.0,0.00,0.00,0.00,0.48,-105.07,0.000000,0.000,0.00000


In [14]:
print("========== DATA QUALITY AUDIT ==========\n")

# 1. DATA TYPES — is every column the type it should be?
print("1. DTYPES:")
print(vendor_summary.dtypes)

# 2. DUPLICATES — full rows, and repeated Vendor+Brand keys
print("\n2. DUPLICATES:")
print("   Full-row duplicates:", vendor_summary.duplicated().sum())
print("   Duplicate Vendor+Brand keys:", vendor_summary.duplicated(subset=['VendorNumber','Brand']).sum())

# 3. MISSING VALUES
print("\n3. MISSING VALUES:", vendor_summary.isnull().sum().sum(), "total")

# 4. WHITESPACE — hidden spaces in text columns (very common in raw data!)
print("\n4. WHITESPACE in text columns:")
for col in ['VendorName', 'Description']:
    n = (vendor_summary[col] != vendor_summary[col].str.strip()).sum()
    print(f"   {col}: {n} values with leading/trailing spaces")

# 5. SANITY — any impossible values?
print("\n5. SANITY CHECKS:")
print("   Negative purchase $:", (vendor_summary['TotalPurchaseDollars'] < 0).sum())
print("   Negative sales $:", (vendor_summary['TotalSalesDollars'] < 0).sum())
print("   GrossProfit < 0 (valid, but worth knowing):", (vendor_summary['GrossProfit'] < 0).sum())

========== DATA QUALITY AUDIT ==========

1. DTYPES:
VendorNumber               int64
VendorName                   str
Brand                      int64
Description                  str
PurchasePrice            float64
ActualPrice              float64
Volume                       str
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalExciseTax           float64
FreightCost              float64
GrossProfit              float64
ProfitMargin             float64
StockTurnover            float64
SalesToPurchaseRatio     float64
dtype: object

2. DUPLICATES:
   Full-row duplicates: 0
   Duplicate Vendor+Brand keys: 0

3. MISSING VALUES: 0 total

4. WHITESPACE in text columns:
   VendorName: 9137 values with leading/trailing spaces
   Description: 0 values with leading/trailing spaces

5. SANITY CHECKS:
   Negative purchase $: 0
   Negative sales $: 0
   GrossProfit < 0 (valid

In [15]:
# FIX 1: Volume — convert text to a real number
vendor_summary['Volume'] = pd.to_numeric(vendor_summary['Volume'], errors='coerce')

# FIX 2: strip hidden spaces from text columns
vendor_summary['VendorName']  = vendor_summary['VendorName'].str.strip()
vendor_summary['Description'] = vendor_summary['Description'].str.strip()

# Re-save the CLEANED table back to the database (overwrite the dirty one)
vendor_summary.to_sql('vendor_sales_summary', con=engine, if_exists='replace', index=False)

# Verify the fixes worked
print("Volume dtype now:", vendor_summary['Volume'].dtype)
print("VendorName spaces remaining:", (vendor_summary['VendorName'] != vendor_summary['VendorName'].str.strip()).sum())
print("✅ Cleaned table re-saved")

Volume dtype now: float64
VendorName spaces remaining: 0
✅ Cleaned table re-saved
